<a href="https://colab.research.google.com/github/eyadarshad/FlyrankAssignment/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eyadarshad/FlyrankAssignment/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [1]:
# LANE: Refresh / Content Opportunity Scoring
# TASK TYPE: Ranking (with binary classification as the scoring engine)
#
# Why ranking, not just classification:
# The end goal isn't to label every page "declining" or "not declining" —
# it's to produce a RANKED LIST so a content team reviews the most urgent
# pages first. Classification (is this page declining? yes/no) feeds the
# ranking, but the deliverable is an ordered queue, not a label.
#
# How it works:
# 1. Train a classifier to estimate P(declining) for each page
# 2. Sort pages by that probability (highest first)
# 3. The top-K pages become the "refresh review queue"
# 4. Attach reason codes explaining WHY each page was flagged
#
# This is a supervised learning problem because we have a label
# (trend_direction == "down") derived from observed data.

print("Task type: Ranking (powered by binary classification)")
print("Lane: Refresh / Content Opportunity Scoring")

Task type: Ranking (powered by binary classification)
Lane: Refresh / Content Opportunity Scoring


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [2]:
# TARGET: is_declining_label (binary: 1 = declining, 0 = not)
#
# Where it comes from:
# The starter dataset has a column called 'trend_direction' with values
# like "up", "down", "stable". The label is defined as:
#   is_declining_label = (trend_direction == "down")
#
# This is an OBSERVED OUTCOME, not a product decision — it's derived
# from the actual measured trend in impressions over two 30-day windows.
#
# It's a PROXY, not ground truth:
# - It's coarse: a page declining 1% and one declining 50% both get label=1
# - It captures direction but not magnitude or urgency
# - It doesn't tell us WHETHER the page was refreshed or what happened after
#
# But it's the best target we have that doesn't leak:
# - trend_direction and trend_pct CANNOT be features (they ARE the label)
# - Product flags (health_score, priority_score) aren't in our data
# - All features must be things knowable BEFORE the decline happened

print("Target: is_declining_label = (trend_direction == 'down')")
print("Source: observed outcome from measured impression trends")
print("Type: binary proxy — captures direction, not magnitude")

Target: is_declining_label = (trend_direction == 'down')
Source: observed outcome from measured impression trends
Type: binary proxy — captures direction, not magnitude


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [3]:
# SUCCESS METRIC: Precision@K (primary), with AUC-ROC as secondary
#
# Why Precision@K:
# The content team can review maybe 20-50 pages per sprint. They don't
# care about the model's accuracy on all 30,000 pages — they care about
# whether the TOP of the list is right. If we flag 50 pages and 40 are
# actually declining, that's Precision@50 = 0.80 — useful.
# If only 10 are right, that's 0.20 — waste of their time.
#
# Why NOT accuracy:
# ~54% of pages are declining (from our ML-02 analysis). A model that
# predicts "declining" for everything gets 54% accuracy — sounds okay,
# but it's useless as a ranking. Precision@K measures what actually
# matters: is the top of the queue trustworthy?
#
# Secondary metric: AUC-ROC
# Measures overall ranking quality across all thresholds. Useful for
# model comparison but doesn't directly tell you "are the top 50 good?"
#
# Baseline to beat:
# The hand-written rule (stale × visible) — whatever its Precision@50 is,
# the model needs to beat it to justify its existence.

print("Primary metric: Precision@K (K = 20, 50)")
print("Secondary metric: AUC-ROC")
print("Baseline: hand-written stale × visible rule")

Primary metric: Precision@K (K = 20, 50)
Secondary metric: AUC-ROC
Baseline: hand-written stale × visible rule


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# UNIT OF ANALYSIS: one content page
# Each row = one page for one client, with 90-day aggregated metrics
print(f"Unit of analysis: one content page")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Each row = one page, identified by content_id, belonging to one client_id")
print(f"Clients: {df['client_id'].nunique()}")
print(f"Target distribution: {df['is_declining_label'].mean():.1%} declining\n")

# Show what one row looks like — the features a model would see
feature_cols = ["content_age_days", "days_since_last_update", "impressions_90d",
                "avg_position", "ctr", "word_count", "engagement_rate",
                "clicks_90d", "sessions_90d"]

print("Sample rows — what the model sees (features) vs what it predicts (target):\n")
sample = df[feature_cols + ["is_declining_label"]].head(5)
print(sample.to_string(index=False))

# Sketch what the TARGET column looks like
print(f"\nTarget column breakdown:")
print(df["is_declining_label"].value_counts().rename({0: "not declining", 1: "declining"}).to_string())

Unit of analysis: one content page
Shape: 30,000 rows × 45 columns
Each row = one page, identified by content_id, belonging to one client_id
Clients: 32
Target distribution: 54.2% declining

Sample rows — what the model sees (features) vs what it predicts (target):

 content_age_days  days_since_last_update  impressions_90d  avg_position  ctr  word_count  engagement_rate  clicks_90d  sessions_90d  is_declining_label
              187                      20             3803          10.6 0.76      3221.0             5.88          29            17                   1
              445                      25            15320          20.3 0.05      2481.0             0.00           7             9                   1
              141                      20            12581          36.5 0.09      3515.0             0.00          11            11                   1
              463                      22            11751           6.2 0.49         NaN             1.28          58   

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [5]:
# WHY ML BEATS A FIXED RULE:
#
# The hand-written rule is: stale (>= 180 days since update) AND visible
# (>= 500 impressions) → rank by impressions. It's intuitive but limited:
#
# 1. IT ONLY USES 2 SIGNALS
#    The rule checks staleness and visibility. It ignores position drift,
#    CTR changes, engagement rate, word count, content age, and session
#    patterns. A model can weigh all of these together.
#
# 2. THE THRESHOLDS ARE ARBITRARY
#    Why 180 days? Why 500 impressions? A model learns the thresholds from
#    data instead of guessing. Maybe 120 days matters more for some content
#    types. The rule can't adapt; a model can.
#
# 3. IT CAN'T CAPTURE INTERACTIONS
#    A page with high impressions + low CTR + dropping position is a very
#    different signal than high impressions + stable CTR. The rule treats
#    them the same. A tree-based model naturally captures these interactions.
#
# 4. WE ALREADY SAW THE GAP
#    In notebook 02, a depth-3 decision tree beat the hand rule at
#    Precision@50 — and that was with just 6 features. More features
#    and a random forest should widen the gap.
#
# HOWEVER — the rule has one advantage:
# It's perfectly explainable. A model must earn its complexity by
# meaningfully beating the rule on held-out, client-split data.
# If it doesn't, the rule wins.

# Quick proof: show the rule's blind spots
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
flagged = df[(stale == 1) & (visible == 1)]
missed = df[(stale == 0) & (df["is_declining_label"] == 1)]

print(f"Hand rule flags: {len(flagged):,} pages")
print(f"Declining pages the rule MISSES (not stale but declining): {len(missed):,}")
print(f"That's {len(missed)/df['is_declining_label'].sum():.0%} of all declining pages — invisible to the rule.")
print(f"\nA model that uses position, CTR, and engagement can find these.")

Hand rule flags: 17 pages
Declining pages the rule MISSES (not stale but declining): 16,180
That's 99% of all declining pages — invisible to the rule.

A model that uses position, CTR, and engagement can find these.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.